# Falcon Challenge A V2

Notebook final para optimizar las liberaciones de la Presa Falcon con la corrección de la instancia Large de 52 semanas, reparación automática de factibilidad, QUBO recalibrado y demostración QAOA en Small.

In [ ]:
import sys
import subprocess

required_modules = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("dimod", "dimod"),
    ("neal", "neal"),
    ("qiskit", "qiskit"),
    ("qiskit_aer", "qiskit-aer"),
    ("qiskit_optimization", "qiskit-optimization"),
    ("qiskit_algorithms", "qiskit-algorithms"),
]

missing = []
for module_name, package_name in required_modules:
    try:
        __import__(module_name)
    except Exception:
        missing.append(package_name)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import os
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from falcon_challenge_v2 import (
    load_ibwc_csv,
    build_weekly_dataset,
    select_critical_window,
    compute_benchmark_params,
    compute_srs,
    threshold_rule,
    adaptive_rule,
    repair_schedule,
    classical_simulated_annealing,
    build_qubo_matrix,
    decode_qubo_solution,
    solve_qubo_with_neal,
    solve_qaoa_small,
    instance_levels,
)

np.random.seed(42)
np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", 80)
plt.style.use("seaborn-v0_8-darkgrid")

print(sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
DATA_DIR = Path("/Volumes/Osiris/Hack Cuantico")
OUTPUT_DIR = DATA_DIR / "resultados_v2"
OUTPUT_DIR.mkdir(exist_ok=True)

storage_path = DATA_DIR / "DataSetExport-Total Storage.Web-Daily-ac-ft@08461200-Instantaneous-TCM-20260622185130.csv"
delta_path = DATA_DIR / "DataSetExport-Discharge Total.Change-in-Storage@08461200-Instantaneous-TCM-20260622185956.csv"
release_path = DATA_DIR / "DataSetExport-Discharge.Best Available@08461300-Instantaneous-m^3 s-20260622190542.csv"
conservation_path = DATA_DIR / "DataSetExport-Percentage.Conservation-Web-Telemetry@08461200-Instantaneous-%-20260622190353.csv"

IBWC_SOURCE_URLS = {
    "release": "https://waterdata.ibwc.gov/AQWebportal/Data/DataSet/Chart/Location/08461300/DataSet/Discharge/Best%20Available/Interval/Latest",
    "elevation": "https://waterdata.ibwc.gov/AQWebportal/Data/DataSet/Chart/Location/08461200/DataSet/Reservoir%20Elevation/Web-Daily-m/Interval/Latest",
}


def load_or_request_ibwc(file_map):
    missing = [name for name, path in file_map.items() if not Path(path).exists()]
    if not missing:
        return {
            name: load_ibwc_csv(str(path), value_col)
            for name, (path, value_col) in file_map.items()
        }

    print("Faltan algunos CSV locales:")
    for name in missing:
        print(f"- {name}")
    print("Si estás en Colab, sube ahora todos los CSV cuando se abra el selector.")

    if "google.colab" in sys.modules:
        from google.colab import files

        uploaded = files.upload()
        uploaded_names = set(uploaded.keys())
        loaded = {}
        for name, (path, value_col) in file_map.items():
            local_name = Path(path).name
            if Path(path).exists():
                loaded[name] = load_ibwc_csv(str(path), value_col)
            elif local_name in uploaded_names:
                loaded[name] = load_ibwc_csv(local_name, value_col)
            elif len(uploaded_names) == 1:
                only_name = next(iter(uploaded_names))
                loaded[name] = load_ibwc_csv(only_name, value_col)
            else:
                raise FileNotFoundError(f"No se encontró {local_name} en la subida.")
        return loaded

    raise FileNotFoundError(
        f"Faltan CSV en {DATA_DIR}. En local, coloca los archivos allí. En Colab, súbelos con el selector."
    )


file_map = {
    "storage": (storage_path, "storage_tcm"),
    "delta": (delta_path, "delta_S_tcm"),
    "release": (release_path, "discharge_m3s"),
    "conservation": (conservation_path, "conservation_pct"),
}

loaded_ibwc = load_or_request_ibwc(file_map)
storage_df = loaded_ibwc["storage"]
delta_df = loaded_ibwc["delta"]
release_df = loaded_ibwc["release"]
conservation_df = loaded_ibwc["conservation"]

weekly_data = build_weekly_dataset(storage_df, delta_df, release_df, conservation_df)

print("Storage rows:", len(storage_df))
print("Delta rows:", len(delta_df))
print("Release rows:", len(release_df))
print("Weekly merged rows:", len(weekly_data["weekly"]))

print("IBWC source URLs:")
print("Discharge:", IBWC_SOURCE_URLS["release"])
print("Elevation:", IBWC_SOURCE_URLS["elevation"])

weekly_data["weekly"].head()

In [ ]:
def find_critical_window(weekly_dataset, storage_frame, window_weeks):
    return select_critical_window(weekly_dataset, storage_frame, window_weeks)

window_map = {"Small": 12, "Medium": 26, "Large": 52}
critical_windows = {}

storage_weekly = weekly_data["storage_weekly"].copy()
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(storage_weekly["timestamp"], storage_weekly["storage_tcm"], color="#1f6feb", linewidth=1.2, label="Storage semanal")

for name, horizon in window_map.items():
    critical_windows[name] = find_critical_window(weekly_data, storage_df, horizon)
    window = critical_windows[name]
    start = window["timestamp"].iloc[0]
    end = window["timestamp"].iloc[-1]
    subset = storage_weekly[(storage_weekly["timestamp"] >= start) & (storage_weekly["timestamp"] <= end)]
    ax.axvspan(start, end, alpha=0.10, label=f"{name} ({horizon} semanas)")
    print(f"{name}: {start.date()} -> {end.date()} | semanas = {len(window)}")
    print(window[["timestamp", "R_obs_tcm", "delta_S_tcm"]].head(2))

ax.set_title("Selección programática de la ventana crítica de sequía")
ax.set_xlabel("Fecha")
ax.set_ylabel("Storage (TCM)")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## Implementación de baselines clásicos

Se usan las políticas Threshold y Adaptive, pero ambas se reparan automáticamente antes de evaluar factibilidad para que la restricción binacional del 10% no rompa la solución.

In [ ]:
def step(level_t, inflow_t, release_t, evap_t, s_max):
    next_level = level_t + inflow_t - release_t - evap_t
    return float(np.clip(next_level, 0.0, s_max))


def simulate_policy(u_seq, s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta):
    return compute_srs(u_seq, s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta)


def validator(u_seq, s0, dS_obs, R_obs, s_min, s_max, u_max, eta):
    _, _, feasible, components = compute_srs(u_seq, s0, dS_obs, R_obs, s_min, s_max, u_max, 1.0, 1.0, 1.0, eta)
    return feasible, components["violations"]


def repair_policy(u_seq, R_obs, s0, dS_obs, s_min, s_max, u_max, eta, levels):
    return repair_schedule(u_seq, R_obs, s0, dS_obs, s_min, s_max, u_max, eta, levels)


instance_name = "Medium"
window = critical_windows[instance_name]
params = compute_benchmark_params(storage_df, window, len(window))
levels = instance_levels(params["DELTA_U"], 5)

R_obs = window["R_obs_tcm"].to_numpy(dtype=float)
dS_obs = window["delta_S_tcm"].to_numpy(dtype=float)
s0 = params["S_INIT"]
s_max = params["S_MAX"]
s_min = params["S_MIN"]
u_max = params["U_MAX"]
w1 = params["W1"]
w2 = params["W2"]
w3 = params["W3"]
eta = params["ETA"]

u_hist = np.zeros(len(window), dtype=float)
u_threshold_raw = threshold_rule(s0, dS_obs, R_obs, s_min, params["DELTA_U"], len(window))
u_adaptive_raw = adaptive_rule(s0, dS_obs, R_obs, s_min, s_max, params["DELTA_U"], len(window), levels)

u_threshold = repair_policy(u_threshold_raw, R_obs, s0, dS_obs, s_min, s_max, u_max, eta, levels)
u_adaptive = repair_policy(u_adaptive_raw, R_obs, s0, dS_obs, s_min, s_max, u_max, eta, levels)

for label, schedule in [("Historical", u_hist), ("Threshold", u_threshold), ("Adaptive", u_adaptive)]:
    srs, storage, feasible, components = simulate_policy(schedule, s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta)
    print(label, "SRS =", round(srs, 6), "feasible =", feasible, "violations =", components["violations"][:2])

## Reparación automática y validación de la regla binacional del 10%

La reparación se aplica después de cada baseline y también después de decodificar QUBO/QAOA, para que el reporte final siempre distinga entre solución cruda y solución factible.

In [ ]:
def evaluate_solution(u_seq, name, s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta):
    srs, storage, feasible, components = compute_srs(u_seq, s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta)
    return {
        "name": name,
        "srs": srs,
        "feasible": feasible,
        "storage": storage,
        "components": components,
        "u_seq": np.asarray(u_seq, dtype=float),
    }


print("Validator Historical:", validator(u_hist, s0, dS_obs, R_obs, s_min, s_max, u_max, eta))
print("Validator Threshold:", validator(u_threshold, s0, dS_obs, R_obs, s_min, s_max, u_max, eta))
print("Validator Adaptive:", validator(u_adaptive, s0, dS_obs, R_obs, s_min, s_max, u_max, eta))

## Formulación QUBO inicial con variables one-hot

La formulación usa una variable binaria por nivel y semana. La solución física se recupera con decodificación one-hot y reparación posterior.

In [ ]:
qubo_cache = {}

def build_instance_qubo(name, horizon, n_levels, window, params):
    levels = instance_levels(params["DELTA_U"], n_levels)
    Q, n_vars, qmeta = build_qubo_matrix(
        window["delta_S_tcm"].to_numpy(dtype=float)[:horizon],
        window["R_obs_tcm"].to_numpy(dtype=float)[:horizon],
        params["S_INIT"],
        params["S_MIN"],
        params["S_MAX"],
        params["U_MAX"],
        params["W1"],
        params["W2"],
        params["W3"],
        levels,
        horizon,
        eta=params["ETA"],
    )
    qubo_cache[name] = {"Q": Q, "n_vars": n_vars, "levels": levels, "qmeta": qmeta}
    return Q, n_vars, levels, qmeta

Q_medium, n_medium, levels_medium, qmeta_medium = build_instance_qubo("Medium", len(window), 5, window, params)
print("Q vars:", n_medium)
print("lambda_oh:", qmeta_medium["lambda_oh"])
print("lambda_crit:", qmeta_medium["lambda_crit"])
print("lambda_ah:", qmeta_medium["lambda_ah"])

## Mejora de $C_{crit}$ y recalibración de pesos $\lambda$

Se usa una búsqueda pequeña de factores de escala sobre la formulación mejorada para favorecer soluciones que superen el histórico sin romper la factibilidad.

In [ ]:
lambda_grid = [0.5, 1.0, 1.5]
best_qubo = None
best_qubo_score = -np.inf
best_qubo_meta = None
best_qubo_solution = None

for scale_oh in lambda_grid:
    for scale_crit in lambda_grid:
        for scale_ah in lambda_grid:
            Q_try, n_try, qmeta_try = build_qubo_matrix(
                window["delta_S_tcm"].to_numpy(dtype=float)[:len(window)],
                window["R_obs_tcm"].to_numpy(dtype=float)[:len(window)],
                params["S_INIT"],
                params["S_MIN"],
                params["S_MAX"],
                params["U_MAX"],
                params["W1"],
                params["W2"],
                params["W3"],
                levels_medium,
                len(window),
                lambda_oh=qmeta_medium["lambda_oh"] * scale_oh,
                lambda_crit=qmeta_medium["lambda_crit"] * scale_crit,
                lambda_ah=qmeta_medium["lambda_ah"] * scale_ah,
                eta=params["ETA"],
            )
            sample_try, solver_meta = solve_qubo_with_neal(Q_try, num_reads=120, num_sweeps=3000, seed=42)
            if sample_try is None:
                continue
            u_try = decode_qubo_solution(sample_try, len(window), levels_medium)
            u_try = repair_policy(u_try, R_obs, s0, dS_obs, s_min, s_max, u_max, eta, levels_medium)
            srs_try, _, feasible_try, _ = compute_srs(u_try, s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta)
            score = srs_try if feasible_try else srs_try - 1e6
            if score > best_qubo_score:
                best_qubo_score = score
                best_qubo_solution = u_try
                best_qubo_meta = {**qmeta_try, **solver_meta, "scale_oh": scale_oh, "scale_crit": scale_crit, "scale_ah": scale_ah, "srs": srs_try, "feasible": feasible_try}
                best_qubo = Q_try

print("Best QUBO score:", best_qubo_score)
print("Best QUBO meta:", best_qubo_meta)

## Resolución del QUBO con Simulated Annealing (dimod) y tuning

Se resuelve Medium con SA, se registra el mejor schedule y se compara contra Historical, Threshold y Adaptive.

In [ ]:
sa_result = None
qubo_result = None
qaoa_result = None

start_sa = time.time()
u_sa, srs_sa_raw, sa_history = classical_simulated_annealing(
    dS_obs,
    R_obs,
    s0,
    s_min,
    s_max,
    u_max,
    w1,
    w2,
    w3,
    levels,
    len(window),
    n_iter=40000,
    seed=42,
    eta=eta,
)
sa_runtime = time.time() - start_sa
sa_result = evaluate_solution(u_sa, "Classical SA", s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta)
sa_result["runtime"] = sa_runtime
sa_result["history"] = sa_history

sample_sa, sa_meta = solve_qubo_with_neal(best_qubo if best_qubo is not None else Q_medium, num_reads=200, num_sweeps=5000, seed=42)
if sample_sa is not None:
    u_qubo = decode_qubo_solution(sample_sa, len(window), levels_medium)
    u_qubo = repair_policy(u_qubo, R_obs, s0, dS_obs, s_min, s_max, u_max, eta, levels_medium)
    qubo_result = evaluate_solution(u_qubo, "QUBO + SA", s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta)
    qubo_result["runtime"] = sa_meta.get("runtime", 0.0)
    qubo_result["solver_meta"] = sa_meta

print("SA:", sa_result["srs"], sa_result["feasible"], round(sa_result["runtime"], 2), "s")
if qubo_result is not None:
    print("QUBO+SA:", qubo_result["srs"], qubo_result["feasible"], qubo_result["runtime"], "s")

## Implementación QAOA (Qiskit) para instancia Small

QAOA se ejecuta solo en Small para demostrar la técnica de compuertas y evitar costos de simulación excesivos en Medium/Large.

In [ ]:
small_window = critical_windows["Small"]
small_params = compute_benchmark_params(storage_df, small_window, len(small_window))
small_levels = instance_levels(small_params["DELTA_U"], 3)
small_Q, small_n, small_meta = build_qubo_matrix(
    small_window["delta_S_tcm"].to_numpy(dtype=float),
    small_window["R_obs_tcm"].to_numpy(dtype=float),
    small_params["S_INIT"],
    small_params["S_MIN"],
    small_params["S_MAX"],
    small_params["U_MAX"],
    small_params["W1"],
    small_params["W2"],
    small_params["W3"],
    small_levels,
    len(small_window),
    eta=small_params["ETA"],
)

qaoa_sample, qaoa_meta = solve_qaoa_small(small_Q, small_n, reps=1, maxiter=80, seed=42)
if qaoa_sample is not None:
    u_qaoa = decode_qubo_solution(qaoa_sample, len(small_window), small_levels)
    u_qaoa = repair_policy(
        u_qaoa,
        small_window["R_obs_tcm"].to_numpy(dtype=float),
        small_params["S_INIT"],
        small_window["delta_S_tcm"].to_numpy(dtype=float),
        small_params["S_MIN"],
        small_params["S_MAX"],
        small_params["U_MAX"],
        small_params["ETA"],
        small_levels,
    )
    qaoa_result = evaluate_solution(
        u_qaoa,
        "QAOA (Small)",
        small_params["S_INIT"],
        small_window["delta_S_tcm"].to_numpy(dtype=float),
        small_window["R_obs_tcm"].to_numpy(dtype=float),
        small_params["S_MIN"],
        small_params["S_MAX"],
        small_params["U_MAX"],
        small_params["W1"],
        small_params["W2"],
        small_params["W3"],
        small_params["ETA"],
    )
    qaoa_result["runtime"] = qaoa_meta.get("runtime", 0.0)
    qaoa_result["solver_meta"] = qaoa_meta

print("QAOA available:", qaoa_sample is not None)
if qaoa_result is not None:
    print("QAOA Small:", qaoa_result["srs"], qaoa_result["feasible"], qaoa_result["runtime"], "s")
else:
    print("QAOA fallback:", qaoa_meta)

## Validación automática, métricas y visualizaciones comparativas

Se construye un resumen tabular con factibilidad, mejora frente al histórico y runtime, junto con gráficos de niveles y liberaciones.

In [ ]:
results_table = []
all_results = {
    "Historical": evaluate_solution(u_hist, "Historical", s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta),
    "Threshold": evaluate_solution(u_threshold, "Threshold", s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta),
    "Adaptive": evaluate_solution(u_adaptive, "Adaptive", s0, dS_obs, R_obs, s_min, s_max, u_max, w1, w2, w3, eta),
    "Classical SA": sa_result,
}
if qubo_result is not None:
    all_results["QUBO + SA"] = qubo_result

for method_name, result in all_results.items():
    results_table.append({
        "instance": instance_name,
        "method": method_name,
        "srs": result["srs"],
        "delta_vs_hist": result["srs"] - all_results["Historical"]["srs"],
        "feasible": result["feasible"],
        "runtime_s": result.get("runtime", 0.0),
    })

if qaoa_result is not None:
    results_table.append({
        "instance": "Small",
        "method": "QAOA (Small)",
        "srs": qaoa_result["srs"],
        "delta_vs_hist": np.nan,
        "feasible": qaoa_result["feasible"],
        "runtime_s": qaoa_result.get("runtime", 0.0),
    })

summary_df = pd.DataFrame(results_table)
display(summary_df)

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=False)

plot_df = summary_df[summary_df["instance"] == instance_name].copy()
axes[0].bar(plot_df["method"], plot_df["srs"], color=["#8b949e", "#f0883e", "#a371f7", "#58a6ff", "#3fb950"][:len(plot_df)])
axes[0].set_title(f"SRS por método - {instance_name}")
axes[0].set_ylabel("SRS")
axes[0].tick_params(axis='x', rotation=20)

medium_focus = sa_result
weeks = np.arange(len(medium_focus["storage"]))
axes[1].plot(weeks, medium_focus["storage"], label="Classical SA", linewidth=2, color="#58a6ff")
axes[1].plot(weeks, all_results["Historical"]["storage"], label="Historical", linewidth=1.5, linestyle="--", color="#8b949e")
axes[1].plot(weeks, all_results["Threshold"]["storage"], label="Threshold", linewidth=1.5, color="#f0883e")
axes[1].plot(weeks, all_results["Adaptive"]["storage"], label="Adaptive", linewidth=1.5, color="#a371f7")
if qubo_result is not None:
    axes[1].plot(weeks, qubo_result["storage"], label="QUBO + SA", linewidth=2.2, color="#3fb950")
axes[1].axhline(s_min, color="#f85149", linestyle=":", label="S_min")
axes[1].set_title("Trayectorias de almacenamiento reparadas")
axes[1].set_xlabel("Semana")
axes[1].set_ylabel("Storage (TCM)")
axes[1].legend(ncol=3, fontsize=9)

plt.tight_layout()
plt.show()

## Empaquetado para Colab/VSCode, tests unitarios y reproducibilidad

Comandos recomendados: ejecutar el notebook completo, exportarlo a HTML con nbconvert y correr pruebas locales para validator, repair_schedule y el simulador.

In [ ]:
print("Para exportar el notebook a HTML:")
print("python -m nbconvert --to html falcon_challenge_A_final_v2.ipynb")
print()
print("Pruebas sugeridas:")
print("pytest -q tests/test_validator.py tests/test_repair.py tests/test_simulator.py")
print()
print("Semillas fijadas en 42 para NumPy y los solvers configurados en el notebook.")

## Cierre

La versión V2 deja el benchmark listo con Large a 52 semanas, reparación automática de factibilidad, QUBO mejor calibrado y una demostración QAOA en Small. El siguiente paso es ejecutar el notebook completo y guardar las tablas finales y figuras exportadas.